### Feature Engineering

Feature engineering was applied to create additional variables that may improve the predictive performance of the models.

The goal of this step is to capture hidden relationships between existing features and generate more informative variables for machine learning algorithms. New features were created using customer tenure, billing information, and service usage patterns.

These engineered features aim to better represent customer behavior and potential churn risk.

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

In [27]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [29]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [30]:
if "customerID" in df.columns:
    df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)


In [31]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [32]:
df['Churn'] = df['Churn'].map({'No':0, 'Yes':1})

In [34]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [35]:
df['Average_Monthly_Charge'] = np.where(df['tenure'] == 0, 
                                        df['MonthlyCharges'], 
                                        df['TotalCharges'] / df['tenure'])
df['Charge_Difference'] = df['MonthlyCharges'] - df['Average_Monthly_Charge']


df['Is_Streaming_User'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
df['Security_and_Support_Risk'] = ((df['OnlineSecurity'] == 'No') & (df['TechSupport'] == 'No')).astype(int)
df['Is_Family'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).astype(int)
df['Is_AutoPay'] = df['PaymentMethod'].str.contains('automatic').astype(int)


df['Risk_Segment'] = ((df['Contract'] == 'Month-to-month') & (df['PaymentMethod'] == 'Electronic check')).astype(int)
df['short_contract_low_tenure'] = ((df['Contract'] == 'Month-to-month') & (df['tenure'] < 12)).astype(int)

contract_map = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2
}
df["contract_length"] = df["Contract"].map(contract_map)


df = df.drop('Contract', axis=1)


bins = [-1, 12, 36, 60, np.inf]
labels = ['0-1 Year', '1-3 Years', '3-5 Years', '5+ Years']
df['Tenure_Cohorts'] = pd.cut(df['tenure'], bins=bins, labels=labels)


service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", 
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
df["total_services"] = (df[service_cols] == "Yes").sum(axis=1)
df["charge_per_service"] = df["MonthlyCharges"] / (df["total_services"] + 1)
df["has_internet"] = (df["InternetService"] != "No").astype(int)


df['senior_high_charge'] = ((df['SeniorCitizen'] == 1) & (df['MonthlyCharges'] > df['MonthlyCharges'].median())).astype(int)

In [37]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,Is_Family,Is_AutoPay,Risk_Segment,short_contract_low_tenure,contract_length,Tenure_Cohorts,total_services,charge_per_service,has_internet,senior_high_charge
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,1,0,1,1,0,0-1 Year,1,14.9250,1,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,0,0,0,0,1,1-3 Years,3,14.2375,1,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,0,0,0,1,0,0-1 Year,3,13.4625,1,0
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,0,1,0,0,1,3-5 Years,3,10.5750,1,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,0,0,1,1,0,0-1 Year,1,35.3500,1,0


Rare Category Handling

In [38]:
categorical_cols = df.select_dtypes("object").columns.tolist()

df= pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

In [39]:
print("Final feature count:", df.shape[1])

Final feature count: 43


In [40]:
X=df.drop(["Churn"], axis=1)
y=df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [41]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (5634, 42)
X_test: (1409, 42)


#### Feature Engineering Results

After applying feature engineering, the number of features increased from **21 to 43**, allowing the dataset to capture more detailed information about customer behavior and service usage.

The dataset still contains **7043 observations**, but the expanded feature set provides additional signals that may help machine learning models detect churn patterns more effectively.

In [42]:
X_train.to_csv("../data/X_train_fe.csv", index=False)
X_test.to_csv("../data/X_test_fe.csv", index=False)

y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

In [43]:
import joblib

joblib.dump(list(X_train.columns),"../data/feature_columns.pkl")

['../data/feature_columns.pkl']